# Attribute Evaluation (Feature selection)

In machine learning, we can sometimes improve model performance by removing features that do not contain helpful information or by focusing only on the most important features. This can help to:

* Reduce overfitting. A smaller number of features mean models have less chance to overfit on noise present in the training data.
* Improve accuracy. Some models are not good at distinguishing between useful and useless features. By selecting only the most important features, we can sometimes improve performance of such models.
    
This task is often called feature selection and can be automated using different approaches.

In [ ]:
# !pip install datasets

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

# Example 1

In [ ]:
imdb_dataset = load_dataset('stanfordnlp/imdb')
data = pd.DataFrame(imdb_dataset['train'].shuffle(seed=42).select(range(1000)))
data['label'] = data['label'].apply(lambda x: 'pos' if x else 'neg')
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer()  # Use TF-IDF vectorizer for text to numerical feature conversion
X_train_vec = vectorizer.fit_transform(X_train).todense()
X_test_vec = vectorizer.transform(X_test).todense()

X_train_vec.shape

Our tf-idf vector contains a large amount of rows (features) - one for every word. Most features are not related to review scores and will not be useful when training a classification model. Can we determine which features are useful?

## Univariate Feature Selection

Sklearn contains functions that can evaluate whether certain features are useful for predicting a given class. Univariate feature selection methods do this using statistical tests.

Mutual information measures the mutual dependence between two variables. In other words, it tells us whether knowing something about variable X will reduce the uncertainty about variable Y. If X and Y are independent, MI is equal to 0. Higher values indicate some relation between X and Y. 

f score uses analysis of variance to discover feature importances. For a given feature, it compares the distributions across all possible class values. The more the distributions differ between classes, the better the score.

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import f_classif

In [ ]:
# A list of words in our tf-idf matrix
list(vectorizer.vocabulary_)[:10]

In [ ]:
# Evaluate each word with f values
feature_scores = f_classif(np.asarray(X_train_vec), y_train)[0]
feature_scores

In [ ]:
word_scores = {k: v for k, v in zip(vectorizer.get_feature_names_out(), feature_scores)}
sorted_words = sorted(word_scores, key = word_scores.get)

# Print 30 words with highest mutual information
print("Highest")
for i in range(1, 31):
    print(sorted_words[-i], word_scores[sorted_words[-i]])
    
# And 30 lowest:
print("Lowest")
for i in range(0, 30):
    print(sorted_words[i], word_scores[sorted_words[i]])
    

In [ ]:
bad_features_train = X_train_vec[:,feature_scores < 1 ]
bad_features_test = X_test_vec[:,feature_scores < 1]
bad_features_train.shape

In [ ]:
good_features_train = X_train_vec[:,feature_scores > 1 ]
good_features_test = X_test_vec[:,feature_scores > 1]
good_features_train.shape

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# KNN model
knn_model = KNeighborsClassifier()
knn_model.fit(np.asarray(bad_features_train), y_train)
knn_predictions = knn_model.predict(np.asarray(bad_features_test))
knn_accuracy = accuracy_score(y_test, knn_predictions)
print("KNN Accuracy on bad features:", knn_accuracy)

# KNN model
knn_model = KNeighborsClassifier()
knn_model.fit(np.asarray(good_features_train), y_train)
knn_predictions = knn_model.predict(np.asarray(good_features_test))
knn_accuracy = accuracy_score(y_test, knn_predictions)
print("KNN on good features:", knn_accuracy)


Good features perform better than bad features, but are they better than all features?

In [ ]:
# KNN model
knn_model = KNeighborsClassifier()
knn_model.fit(np.asarray(X_train_vec), y_train)
knn_predictions = knn_model.predict(np.asarray(X_test_vec))
knn_accuracy = accuracy_score(y_test, knn_predictions)
print("KNN Accuracy on all features:", knn_accuracy)

Lets evaluate different feature subsets

In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

In [ ]:
for n_feats in range(100, X_train_vec.shape[1], 1000):
    feature_selector = SelectKBest(f_classif, k=n_feats)
    X_train_n_best = feature_selector.fit_transform(np.asarray(X_train_vec), y_train)
    X_test_n_best = feature_selector.transform(np.asarray(X_test_vec))
    
    knn_model = KNeighborsClassifier()
    knn_model.fit(X_train_n_best, y_train)
    knn_predictions = knn_model.predict(X_test_n_best)
    knn_accuracy = accuracy_score(y_test, knn_predictions)
    print(n_feats, knn_accuracy)

Certain subsets of features improve performance over all features! This depends on the data and type of model, but feature selection can sometimes be an important part of classification.

# Example 2 - Dependent Features

The previous features assume the features are indepentend. In real-world scenarions, this is not always true and can make such methods innefective. For example, consider the following artificial example:

In [ ]:
quadrant = pd.read_csv("quadrant.txt")
quadrant

"Quadrant" is a dummy dataset where:

* Every feature is, by itself, independent from the class variable. We cannot predict the class based on 1 feature alone
* The only way to make correct predictions is by looking at both a1 and a2.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
colors = ['red' if x else 'blue' for x in quadrant['Class']]
plt.scatter(quadrant['a1'],[1 for _ in quadrant['a1']], color=colors)

In [ ]:
plt.scatter(quadrant['a2'],[1 for _ in quadrant['a1']], color=colors)

In [ ]:
colors = ['red' if x else 'blue' for x in quadrant['Class']]
plt.scatter(quadrant['a2'], quadrant ['a3'], color=colors)

In [ ]:
plt.scatter(quadrant['a4'], quadrant ['a5'], color=colors)

In [ ]:
plt.scatter(quadrant['a1'], quadrant ['a2'], color=colors)

Measures that assume features are independent will fail to properly evaluate features

In [ ]:
mutual_info_classif(quadrant.drop('Class', axis=1), quadrant['Class'])

In [ ]:
f_classif(quadrant.drop('Class', axis=1), quadrant['Class'])[0]

Models using the selected measures cannot classify examples (unless the feature selection methods get lucky and guess the correct features)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

train_x, test_x, train_y, test_y = train_test_split(quadrant.drop('Class', axis=1), quadrant['Class'])

feature_selector = SelectKBest(mutual_info_classif, k=2)
quadrant_2_feats_train = feature_selector.fit_transform(train_x, train_y)
colors = ['red' if x else 'blue' for x in train_y]
plt.scatter([x[0] for x in quadrant_2_feats_train], [x[1] for x in quadrant_2_feats_train], color=colors )

In [ ]:
quadrant_2_feats_test = feature_selector.transform(test_x)
dt_model = DecisionTreeClassifier()
dt_model.fit(quadrant_2_feats_train, train_y)
dt_predictions = dt_model.predict(quadrant_2_feats_test)
dt_accuracy = accuracy_score(test_y, dt_predictions)
dt_accuracy

Instead, we can use methods that take feature dependency into account

In [ ]:
#!pip install sklearn_relief

In [ ]:
import sklearn_relief as relief

In [ ]:
r = relief.Relief(n_features=2, n_jobs=1)

In [ ]:
r.fit(np.asarray(train_x), np.asarray(train_y))
relief_2_feats_train = r.transform(np.asarray(train_x))
relief_2_feats_test = r.transform(np.asarray(test_x))

In [ ]:
print("Relief feature weights")
print(r.w_)

The first two features are correctly identified as important! Relief takes into account feature dependencies and correctly identifies the first two as most useful.

In [ ]:
dt_model = DecisionTreeClassifier()
dt_model.fit(relief_2_feats_train, train_y)
dt_predictions = dt_model.predict(relief_2_feats_test)
dt_accuracy = accuracy_score(test_y, dt_predictions)
dt_accuracy

# Example 3 - Model-based feature selection

Some models already evaluate features during training. For example, decision trees and random forests can assign importances to features that are used to split the trees. After training, we can obtain these features and use them in other models or use them to explain how a model obtained its predictions.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
X_train_vec

In [ ]:
logistic_model = RandomForestClassifier()
logistic_model.fit(np.asarray(X_train_vec), y_train)

for score, name in list(sorted(zip(logistic_model.feature_importances_, vectorizer.get_feature_names_out()), reverse=True))[:10]:
    print(name, score)

# Example 4 - Recursive feature elimination

Recursive feature elimination (RFE) tries to select a subset of best features by recursively removing unhelpful features. It trains a model multiple times, removing a certain amount least-helpful features (according to model-based feature importance) after each iteration.

In [ ]:
 from sklearn.feature_selection import RFE

In [ ]:
estimator = RandomForestClassifier()

In [ ]:
selector = RFE(estimator, n_features_to_select = 12000, step=500, verbose=1)
selector = selector.fit(np.asarray(X_train_vec), y_train)

In [ ]:
X_train_rfe = selector.transform(np.asarray(X_train_vec))
X_test_rfe = selector.transform(np.asarray(X_test_vec))

estimator = RandomForestClassifier()
estimator.fit(np.asarray(X_train_vec), y_train)
preds = estimator.predict(np.asarray(X_test_vec))
score_vec = accuracy_score(y_test, preds)
print("Random forest score on all features:", score_vec)


estimator = RandomForestClassifier()
estimator.fit(X_train_rfe, y_train)
preds = estimator.predict(X_test_rfe)
score_rfe = accuracy_score(y_test, preds)
print("Random forest score on RFE features:", score_rfe)